# GENE7 morphology in the reference morphospace

Places the **GENE7 (20250612)** crispant/temperature panel in the legacy VAE morphospace, against a
wildtype reference trajectory fit from timelapse data plus the **20240813** hotfish plates.

Encoding throughout: **colour = rearing temperature**, **marker symbol = crispant target**,
**black curve = wildtype reference spline**.

Processing lives in companion scripts; this notebook loads cached artifacts and plots.

| script | role |
|---|---|
| `legacy_reference.py` | reference-set QC + the 20240813 temperature repair |
| `morph_pca_spline.py` | PCA basis + reference spline fitting |
| `plotting.py` | figure grammar |
| `run_processing.py` | runs the chain, writes `data/` |

Regenerate the cache with:

```bash
PYTHONPATH=/net/trapnell/vol1/home/nlammers/projects/repositories/morphseq/src:. \
  /net/trapnell/vol1/home/nlammers/micromamba/envs/points-ml/bin/python run_processing.py
```

## Provenance and caveats

- **Morphology is the LEGACY embeddings** (`20241107_ds_sweep01_optimum`), not the current
  pipeline's. The pipeline's snip raster is degraded relative to the build that trained this
  checkpoint (`target_pixel_size_um` 7.8 vs 6.5, plus a saturation difference), so its latents are
  not interchangeable.
- **No QC from the current pipeline.** Its `use_snip` / `sa_outlier_flag` were computed on that
  degraded raster and against a reference population that mis-calibrates cold-reared embryos.
  Exclusions are curated by inspection.
- **No polynomial surface / `mdl_stage_hpf`.** The 2025-Q1 notebook fit a degree-2 regression from
  PCA space to stage; omitted here by request.
- **GENE7 is projected, never fit.** The PCA basis and the spline come from reference + 20240813
  only, so GENE7 cannot reshape the axes it is measured against.

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parents[2] / "src"))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio

import plotting as pl

pio.renderers.default = "notebook"
DATA = HERE / "data"
FIGS = HERE / "figures"
FIGS.mkdir(exist_ok=True)

## Load cached artifacts

In [2]:
reference = pd.read_csv(DATA / "reference_pca.csv")
hotfish   = pd.read_csv(DATA / "hotfish_pca.csv")
gene7     = pd.read_csv(DATA / "gene7_pca.csv")
variance  = pd.read_csv(DATA / "pca_variance.csv")

splines = {
    "weighted":   pd.read_csv(DATA / "spline_weighted.csv"),
    "unweighted": pd.read_csv(DATA / "spline_unweighted.csv"),
}

print(f"reference : {len(reference):6d} snips / {reference.embryo_id.nunique()} embryos")
print(f"20240813  : {len(hotfish):6d} snips   temps {sorted(hotfish.temperature.unique())}")
print(f"GENE7     : {len(gene7):6d} wells   temps {sorted(gene7.temperature.unique())}")

reference :  14200 snips / 180 embryos
20240813  :    141 snips   temps [19.0, 25.0, 28.5, 32.0, 33.5, 35.0]
GENE7     :    567 wells   temps [24, 28, 34, 35]


### PCA variance

The basis is fit on `z_mu_b_*` (the 80 biological latents) pooled over reference + 20240813.
The nuisance dimensions `z_mu_n_*` are excluded, as in the original notebooks.

In [3]:
display(variance.round(4))
print(f"first 3 PCs: {variance.cumulative.iloc[2]:.1%} of variance -> 3D plots are well justified")

,component,explained_variance_ratio,cumulative
0,1,0.3962,0.3962
1,2,0.2908,0.6870
2,3,0.1639,0.8509
3,4,0.0937,0.9446
4,5,0.0366,0.9812


first 3 PCs: 85.1% of variance -> 3D plots are well justified


## Reference spline variants

Both are local principal curves (50 bootstraps x 1000 resampled points, averaged).

- **unweighted** - reference + 20240813@28.5C pooled, uniform sampling.
- **weighted** - same set, but the 24 control-arm snips are up-weighted to 25% of the sampling mass
  (`ALPHA = 0.25`). Without this they are invisible against ~14,200 reference snips. The original
  notebook introduced this because the reference trajectory diverges from the 28.5C cohort near
  24 hpf, distorting the other temperature arms.

`ALPHA` is acknowledged as ad-hoc in the source notebook; both curves are shown so its effect is
visible rather than assumed.

In [4]:
P = list(pl.PCA_COLUMNS)
divergence = np.sqrt(
    ((splines["weighted"][P].to_numpy() - splines["unweighted"][P].to_numpy()) ** 2).sum(-1)
)
print(f"mean 5D separation between the two curves: {divergence.mean():.3f}")
print(f"max                                       : {divergence.max():.3f}")

mean 5D separation between the two curves: 0.129
max                                       : 0.361


## 2D PCA

In [5]:
fig = pl.plot_pca_2d(gene7, splines, dims=(0, 1), reference=reference)
pl.save_figure(fig, FIGS / "gene7_pca_2d_pc12")
fig.show()

  [note] PNG export skipped for gene7_pca_2d_pc12: ValueError


In [6]:
fig = pl.plot_pca_2d(gene7, splines, dims=(0, 2), reference=reference,
                     title="GENE7 morphology PCA vs wildtype reference (PC1 vs PC3)")
pl.save_figure(fig, FIGS / "gene7_pca_2d_pc13")
fig.show()

  [note] PNG export skipped for gene7_pca_2d_pc13: ValueError


## 3D PCA

In [7]:
fig = pl.plot_pca_3d(gene7, splines, dims=(0, 1, 2), reference=reference)
pl.save_figure(fig, FIGS / "gene7_pca_3d")
fig.show()

  [note] PNG export skipped for gene7_pca_3d: ValueError


### 3D without the reference backdrop

The reference cloud is dense; this view shows GENE7 and the curve alone.

In [8]:
fig = pl.plot_pca_3d(gene7, splines, dims=(0, 1, 2), reference=None,
                     title="GENE7 only, vs reference spline (3D)")
pl.save_figure(fig, FIGS / "gene7_pca_3d_nobackdrop")
fig.show()

  [note] PNG export skipped for gene7_pca_3d_nobackdrop: ValueError


## Displacement from the reference trajectory

Distance to the weighted curve in 5D PCA space. This is a summary statistic, not a staging estimate:
without the polynomial surface there is no per-embryo morphological stage, so this measures *how far
from the wildtype trajectory* an embryo sits, not *where along it*.

In [9]:
for frame in (gene7, hotfish):
    frame["dist_to_spline"] = pl.distance_to_spline(frame, splines["weighted"])

summary = pd.concat([
    gene7.assign(cohort="GENE7").groupby(["cohort", "temperature"])["dist_to_spline"]
        .agg(["count", "mean", "median"]),
    hotfish.assign(cohort="20240813").groupby(["cohort", "temperature"])["dist_to_spline"]
        .agg(["count", "mean", "median"]),
])
display(summary.round(3))

count   mean  median
cohort   temperature                      
GENE7    24.0           142  1.589   1.582
         28.0           142  1.427   1.294
         34.0           144  1.442   1.163
         35.0           139  2.879   3.043
20240813 19.0            24  0.343   0.303
         25.0            22  0.373   0.329
         28.5            24  0.227   0.219
         32.0            24  0.192   0.146
         33.5            23  0.380   0.303
         35.0            24  1.047   1.081

In [10]:
g = pl.add_perturbation_column(gene7)
fig = px.box(g, x="perturbation_group", y="dist_to_spline",
             color=g.temperature.astype(str),
             category_orders={"perturbation_group": sorted(g.perturbation_group.unique())},
             title="Displacement from the reference trajectory, by perturbation and temperature")
fig.update_layout(width=1000, height=600, font=pl.FONT, plot_bgcolor="white",
                  xaxis_title="crispant target",
                  yaxis_title="distance to reference spline (5D)",
                  legend_title="temp (C)")
pl.save_figure(fig, FIGS / "gene7_displacement_box")
fig.show()

  [note] PNG export skipped for gene7_displacement_box: ValueError


## Read-out

Two things the numbers show, both worth checking against the images before being treated as biology:

1. **35 C separates strongly.** GENE7's 35 C arm sits ~2x further from the reference curve than its
   other arms, and 20240813 shows the same pattern at its own 35 C arm. Consistent across two
   independent experiments, so this is likely real heat-stress morphology.

2. **GENE7 carries a systematic offset.** Even its 28 C controls sit ~1.4 from the curve, against
   ~0.23 for 20240813's 28.5 C controls. Candidate explanations, not yet separated: every GENE7 arm
   is crispant-injected (there is no uninjected wildtype arm), a plate/batch effect, and the 0.5 C
   difference between the two "control" temperatures. Worth resolving before treating absolute
   displacement as comparable across experiments.